# 🐼 Clean Code - Session 2
## Audit et analyse propre avec Pandas

**Objectifs de la session :**
- Réaliser un audit complet de données
- Gérer les valeurs manquantes
- Maîtriser la sélection et le filtrage avancés
- Utiliser les expressions régulières
- Présenter proprement les résultats

---

In [ ]:
# Imports
import pandas as pd
import numpy as np
import re

# Configuration de l'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

## 1. 📥 Chargement du dataset

Nous travaillons sur un dataset de campagne marketing e-commerce avec des problèmes de qualité intentionnels.

In [ ]:
# Chargement des données
df = pd.read_csv('../data/campagne_marketing_ecommerce.csv')

print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

---
## 2. 🔍 Audit initial

### 2.1 Vue d'ensemble

In [ ]:
# Exercice : Afficher les informations générales du DataFrame
# Utilisez la méthode appropriée

df.____()

In [ ]:
# Exercice : Afficher les premières lignes
df.____(10)

In [ ]:
# Exercice : Afficher les statistiques descriptives
df.____(include='all')

### 2.2 Analyse des types de données

In [ ]:
# Exercice : Afficher les types de chaque colonne
df.____

In [ ]:
# Question : Quelles colonnes ont un type incorrect ?
# Notez vos observations :

# Colonnes à convertir :
# - ____ devrait être de type ____
# - ____ devrait être de type ____
# - ____ devrait être de type ____

### 2.3 Analyse des valeurs manquantes

In [ ]:
# Exercice : Calculer le nombre de valeurs manquantes par colonne
missing = df.____().____
print(missing)

In [ ]:
# Exercice : Calculer le pourcentage de valeurs manquantes
missing_pct = (df.____.____ / len(df) * 100).round(2)
print(missing_pct.sort_values(ascending=False))

In [ ]:
# Exercice : Créer un rapport d'audit des valeurs manquantes

def audit_missing_values(dataframe):
    """
    Génère un rapport sur les valeurs manquantes d'un DataFrame.
    
    Parameters
    ----------
    dataframe : pd.DataFrame
        DataFrame à auditer.
    
    Returns
    -------
    pd.DataFrame
        Rapport avec colonnes : missing_count, missing_pct, dtype.
    """
    report = pd.DataFrame({
        'missing_count': ____,
        'missing_pct': ____,
        'dtype': ____
    })
    return report.sort_values('missing_pct', ascending=False)

# Testez votre fonction
audit_missing_values(df)

### 2.4 Détection des doublons

In [ ]:
# Exercice : Compter les doublons exacts
n_duplicates = df.____().sum()
print(f"Nombre de lignes dupliquées : {n_duplicates}")

In [ ]:
# Exercice : Trouver les doublons basés sur certaines colonnes
# (même email + même date + même produit)
cols_key = ['client_email', 'date_transaction', 'produit']
n_duplicates_key = df.duplicated(subset=____).sum()
print(f"Doublons potentiels (email+date+produit) : {n_duplicates_key}")

In [ ]:
# Exercice : Afficher quelques exemples de doublons
duplicates_mask = df.duplicated(subset=cols_key, keep=False)
df[____].sort_values(cols_key).head(10)

---
## 3. 🧹 Gestion des valeurs manquantes

### 3.1 Analyse par colonne

In [ ]:
# Analyse de la colonne 'nb_visites_avant_achat'
# Cette colonne est numérique avec des valeurs manquantes

print("Statistiques de nb_visites_avant_achat :")
print(df['nb_visites_avant_achat'].____())

In [ ]:
# Exercice : Imputer les valeurs manquantes avec la médiane
median_visites = df['nb_visites_avant_achat'].____

df['nb_visites_avant_achat_clean'] = df['nb_visites_avant_achat'].fillna(____)

print(f"Médiane utilisée : {median_visites}")
print(f"Valeurs manquantes après imputation : {df['nb_visites_avant_achat_clean'].isna().sum()}")

In [ ]:
# Exercice : Créer un flag pour les valeurs qui étaient manquantes
df['visites_was_missing'] = df['nb_visites_avant_achat'].____

print(f"Nombre de valeurs imputées : {df['visites_was_missing'].sum()}")

### 3.2 Gestion des valeurs "fausses nulles"

In [ ]:
# Certaines colonnes ont des valeurs comme '', 'N/A', 'n/a' qui sont en fait des nulles

# Exercice : Identifier les fausses valeurs nulles dans 'score_satisfaction'
print(df['score_satisfaction'].value_counts(dropna=False).head(15))

In [ ]:
# Exercice : Remplacer les fausses nulles par de vraies NaN
false_nulls = ['', 'N/A', 'n/a', 'NA', 'null', 'None']

df['score_satisfaction_clean'] = df['score_satisfaction'].replace(____, np.nan)

# Vérification
print(df['score_satisfaction_clean'].value_counts(dropna=False).head(10))

In [ ]:
# Exercice : Convertir en numérique (gérer les valeurs comme '4.5' et '3,5')

df['score_satisfaction_clean'] = (
    df['score_satisfaction_clean']
    .str.replace(',', '.')  # Remplacer virgule par point
    .____  # Convertir en numérique
)

print(df['score_satisfaction_clean'].describe())

---
## 4. 🎯 Sélection et filtrage avancés

### 4.1 Filtrage avec `.loc[]`

In [ ]:
# Exercice : Sélectionner les transactions de plus de 500€ pour la catégorie 'Informatique'
# Note : montant_ttc est actuellement de type string avec des formats variés

# D'abord, nettoyons la colonne montant
def clean_amount(value):
    """
    Nettoie et convertit une valeur de montant en float.
    Gère les formats : '123.45', '123,45', '123€', 'EUR 123', etc.
    """
    if pd.isna(value) or value in ['', 'N/A']:
        return np.nan
    
    # Convertir en string et nettoyer
    value = str(value)
    value = value.replace('€', '').replace('EUR', '').replace(' ', '')
    value = value.replace(',', '.')
    
    try:
        return float(value)
    except ValueError:
        return np.nan

df['montant_clean'] = df['montant_ttc'].apply(clean_amount)
print(df['montant_clean'].describe())

In [ ]:
# Exercice : Maintenant, filtrez avec .loc[]
# Transactions > 500€ ET catégorie 'Informatique'

high_value_info = df.loc[
    (df['____'] > 500) & (df['____'] == 'Informatique'),
    ['id_transaction', 'produit', 'montant_clean', 'categorie']
]

print(f"Nombre de transactions : {len(high_value_info)}")
high_value_info.head()

### 4.2 Filtrage avec `.query()`

In [ ]:
# Exercice : Même filtre mais avec .query()
# Plus lisible pour des conditions complexes

result = df.query("____ > 500 and ____ == 'Informatique'")
print(f"Résultat : {len(result)} lignes")

In [ ]:
# Exercice : Utiliser une variable dans query()
min_amount = 1000
categories = ['Informatique', 'Téléphonie']

result = df.query("montant_clean > @____ and categorie in @____")
print(f"Transactions > {min_amount}€ en {categories} : {len(result)}")

### 4.3 Chaînage de méthodes (Method Chaining)

In [ ]:
# Exercice : Écrire un pipeline complet avec chaînage
# 1. Filtrer les montants > 100€
# 2. Grouper par catégorie
# 3. Calculer la somme et la moyenne
# 4. Trier par somme décroissante

result = (
    df
    .query("____")
    .groupby('____')
    .agg({'montant_clean': ['____', '____']})
    .sort_values(('montant_clean', 'sum'), ascending=____)
)

result

---
## 5. 🔤 Expressions régulières avec Pandas

### 5.1 Nettoyage des numéros de téléphone

In [ ]:
# Analyse des formats existants
print("Exemples de numéros de téléphone :")
print(df['client_telephone'].dropna().sample(10).values)

In [ ]:
# Exercice : Nettoyer les numéros (garder uniquement les chiffres)
# Utiliser str.replace() avec une regex

df['telephone_clean'] = (
    df['client_telephone']
    .str.replace(r'____', '', regex=True)  # Supprimer tout ce qui n'est pas un chiffre
)

print(df['telephone_clean'].dropna().sample(10).values)

In [ ]:
# Exercice : Identifier les numéros invalides (pas 10 chiffres)
# Utiliser str.len() ou str.match()

valid_phone_mask = df['telephone_clean'].str.match(r'^____$', na=False)

print(f"Téléphones valides : {valid_phone_mask.sum()}")
print(f"Téléphones invalides : {(~valid_phone_mask & df['telephone_clean'].notna()).sum()}")

### 5.2 Validation et extraction d'emails

In [ ]:
# Exercice : Identifier les emails invalides
# Un email valide contient : quelquechose@quelquechose.quelquechose

email_pattern = r'^[\w.-]+@[\w.-]+\.[a-zA-Z]{2,}$'

df['email_valid'] = df['client_email'].str.____(____, na=False)

print(f"Emails valides : {df['email_valid'].sum()}")
print(f"Emails invalides : {(~df['email_valid']).sum()}")

In [ ]:
# Exercice : Extraire le domaine des emails
# Utiliser str.extract() avec un groupe de capture

df['email_domain'] = df['client_email'].str.extract(r'@(____)')

# Top 10 des domaines
df['email_domain'].value_counts().head(10)

### 5.3 Extraction d'informations des commentaires

In [ ]:
# Analyse des commentaires
print("Exemples de commentaires :")
for comment in df['commentaire_client'].dropna().sample(10).values:
    print(f"  - {comment}")

In [ ]:
# Exercice : Détecter les commentaires qui mentionnent un numéro de téléphone

has_phone = df['commentaire_client'].str.contains(
    r'____',  # Pattern pour trouver un numéro type 0800 123 456
    na=False,
    regex=True
)

print(f"Commentaires avec numéro de téléphone : {has_phone.sum()}")
print(df.loc[has_phone, 'commentaire_client'].values)

In [ ]:
# Exercice : Extraire les références produit (format SKU-XXXXX-XXX)

df['sku_extracted'] = df['commentaire_client'].str.extract(
    r'(SKU-____-____)'
)

# Afficher les SKU trouvées
df['sku_extracted'].dropna().head(10)

---
## 6. 📊 Statistiques et agrégations

### 6.1 Agrégations groupées

In [ ]:
# Exercice : Calculer des statistiques par catégorie
# - Nombre de transactions
# - Montant total
# - Montant moyen
# - Montant médian

stats_categorie = (
    df
    .groupby('categorie')
    .agg({
        'id_transaction': '____',  # Compter
        'montant_clean': ['____', '____', '____']  # Sum, mean, median
    })
    .round(2)
)

stats_categorie

In [ ]:
# Exercice : Renommer les colonnes du résultat
stats_categorie.columns = ['nb_transactions', 'total_montant', 'montant_moyen', 'montant_median']
stats_categorie = stats_categorie.sort_values('total_montant', ascending=False)
stats_categorie

### 6.2 Tableau croisé dynamique

In [ ]:
# Exercice : Créer un pivot table
# Lignes : catégorie
# Colonnes : canal_acquisition (normalisé en minuscules)
# Valeurs : montant moyen

# D'abord, normalisons le canal
df['canal_clean'] = df['canal_acquisition'].str.____

pivot = pd.pivot_table(
    df,
    values='____',
    index='____',
    columns='____',
    aggfunc='____'
).round(2)

pivot

---
## 7. 🎨 Présenter les résultats

### 7.1 Formatage avec Pandas Styler

In [ ]:
# Exercice : Appliquer un formatage conditionnel
# - Fond vert pour les valeurs élevées
# - Fond rouge pour les valeurs basses

(
    stats_categorie
    .style
    .background_gradient(cmap='____', subset=['total_montant'])
    .format({
        'total_montant': '{:,.0f}€',
        'montant_moyen': '{:,.2f}€',
        'montant_median': '{:,.2f}€'
    })
)

### 7.2 Export propre

In [ ]:
# Exercice : Exporter le rapport en Excel avec mise en forme

# Créer un rapport final
report = {
    'statistiques_categorie': stats_categorie,
    'pivot_canal': pivot,
    'audit_missing': audit_missing_values(df)
}

# Export multi-onglets
with pd.ExcelWriter('rapport_audit.xlsx', engine='openpyxl') as writer:
    for sheet_name, data in report.items():
        data.to_excel(writer, sheet_name=sheet_name)

print("✓ Rapport exporté : rapport_audit.xlsx")

---
## 8. 🎯 Exercice récapitulatif : Pipeline complet

Créez une fonction qui réalise un audit complet et retourne un rapport structuré.

In [ ]:
def audit_dataframe(df, date_col=None, amount_col=None, category_col=None):
    """
    Réalise un audit complet d'un DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame à auditer.
    date_col : str, optional
        Nom de la colonne de date.
    amount_col : str, optional
        Nom de la colonne de montant.
    category_col : str, optional
        Nom de la colonne de catégorie.
    
    Returns
    -------
    dict
        Dictionnaire contenant les résultats de l'audit.
    """
    audit = {}
    
    # 1. Informations générales
    audit['shape'] = {
        'rows': df.shape[0],
        'columns': df.shape[1]
    }
    
    # 2. Valeurs manquantes
    audit['missing'] = {
        'total_cells': df.____.____.____,
        'pct_missing': round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2),
        'by_column': df.isna().sum().to_dict()
    }
    
    # 3. Doublons
    audit['duplicates'] = {
        'exact_duplicates': df.____().____,
        'pct_duplicates': round(df.duplicated().sum() / len(df) * 100, 2)
    }
    
    # 4. Types de données
    audit['dtypes'] = df.dtypes.astype(str).to_dict()
    
    # 5. Statistiques par catégorie (si spécifiée)
    if category_col and amount_col:
        audit['by_category'] = (
            df
            .groupby(category_col)[amount_col]
            .agg(['count', 'sum', 'mean'])
            .round(2)
            .to_dict()
        )
    
    return audit

# Test de la fonction
result = audit_dataframe(
    df, 
    date_col='date_transaction',
    amount_col='montant_clean',
    category_col='categorie'
)

# Affichage structuré
import json
print(json.dumps(result, indent=2, default=str))

---
## ✅ Checklist Clean Code - Projets Data

Avant de livrer votre code, vérifiez :

- [ ] Structure de projet claire (data/, src/, notebooks/)
- [ ] Docstrings sur toutes les fonctions
- [ ] Nommage explicite (df_cleaned, calculate_revenue)
- [ ] Pas de magic numbers (constantes nommées)
- [ ] Audit documenté (% missing, doublons, anomalies)
- [ ] Pipeline reproductible (paramètres, seed)
- [ ] Gestion des erreurs (try/except, validation)
- [ ] Exports formatés et lisibles

---
**🎉 Félicitations ! Vous avez terminé le module Clean Code !**